In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from backend.extract_filter import classify_and_extract
from backend.data_utils import fetch_papers, verify_results, chunk_papers, format_query, clear
from backend.Database import create_index, save_embeddings_to_index, delete_index
from backend.retrieval import answer_query, clear_memory
from backend.pdf_ingest import ingest_uploaded_pdf
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))


from time import sleep

'export' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���
c:\Users\guoji\Group-3\gjwenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from pinecone import Pinecone

# indexes = pc.list_indexes()

# # delete all indexes
# for idx in indexes:
#     index_name = idx["name"]
#     pc.delete_index(index_name)
#     print(f"Deleted index: {index_name}")


In [3]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

Index 'gpy9h451je' created successfully!


In [ ]:
clear_memory()
clear()

def handleUserQuery(user_input, user_index):

    path = 
    ingest_uploaded_pdf(file_path="", user_index=index_obj)

    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)
    print(classification)
 
    filters = classification['filters']
    print("\nRunning Search with filters:", filters)

    # if no academic query present, call llm without fetching paper
    if not format_query(filters):   
        question = classification['question']
        answer = answer_query(classification, index_obj, top_k=100)
        print("Generated Answer:")
        print(answer)
        return

    # Step 1: Fetch papers
    papers = fetch_papers(filters)

    # Step 2: Verify & Filter papers
    verified_papers = verify_results(papers, filters)
    print(f"[integration] Found {len(verified_papers)} verified paper(s).")
    
    # Determine number of papers to return; default to 1 if not provided
    try:
        num_papers = int(filters.get("max_results", 1))
    except ValueError:
        num_papers = 1
    num_papers = min(num_papers, len(verified_papers))

    # Step 3: Chunk papers
    chunks = chunk_papers(verified_papers[:num_papers])    # Get full paper content as chunks
    texts = [chunk["text"] for chunk in chunks]            # Extract just the text
    print("text:")
    print(texts)

    titles = [chunk["title"] for chunk in chunks]    # Extract the titles
    print('titles: ')
    print(titles)

    authors = [[author.lower() for author in chunk["author"]] for chunk in chunks]
    print("author:")
    print(authors)

    dates = [chunk["date"] for chunk in chunks]    # Extract the titles
    print('date: ')
    print(dates)


    if texts:
        # Save the full paper text chunks to the index in batches
        batch_size = 200
        for i in range(0, len(texts), batch_size): 
            print('i: ', i)
            text_batch = texts[i:i+batch_size]
            title_batch = titles[i:i+batch_size]     # 修改此处
            author_batch = authors[i:i+batch_size]    # 修改此处
            date_batch = dates[i:i+batch_size]        # 修改此处
            # 如果 sources 也是列表，也需做切片，否则传入列表或保持一致
            source_batch = ["arxiv"] * len(text_batch)
            save_embeddings_to_index(
                index_name=user_index, 
                text=text_batch, 
                titles=title_batch, 
                authors=author_batch, 
                dates=date_batch, 
                sources=source_batch,
                namespace="default"
            )
            sleep(1)



    #answer search question
    answer = answer_query(classification=classification, index_obj=index_obj, top_k=100)
    print("Generated Answer:")
    print(answer)
 
# Use a loop to enable continuous conversation
while True: 
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        # delete_index(index_name=index_name)
        clear_memory()
        clear()
        break
    handleUserQuery(user_query, index_name)


Memory cleared.
{'action': 'other', 'filters': {'query': 'hello', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}, 'question': 'hello'}

Running Search with filters: {'query': 'hello', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}
search query:  hello
added  The distributed Language Hello White Paper  to prevent searching it next time
[integration] Found 1 verified paper(s).
Extracted 266 chunks from 'The distributed Language Hello White Paper'

Total Chunks Extracted: 266

text:
['September 12, 2014 \n \nHello White Paper \n \n \n \nThe Distributed Language  \n \n \n \n \nV1.0 .3  (alpha ) \n \n \nWhite Paper  \nBoris Burshteyn \nbburshteyn@amsdec.com V1.0.3 (alpha) One So

Token indices sequence length is longer than the specified maximum sequence length for this model (13877 > 1024). Running this sequence through the model will result in indexing errors
c:\Users\guoji\Group-3\backend\retrieval.py:219: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty i

After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retr

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Generated Answer:
content="I'm just a program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 11638, 'total_tokens': 11666, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_86d0290411', 'finish_reason': 'stop', 'logprobs': None} id='run-f532d3ca-99b7-4240-b572-ca7fa207d941-0'
{'action': 'other', 'filters': {'query': 'weather', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}, 'question': 'how is the weather'}

Running Search with filters: {'query': 'weather', 'author': None, 'title': 

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Retrieved chunks: 
[{'text': "'scenes under clear weather conditions.\nDiverse Weathers on Real-World Images.We also visualize the 3D de-\ntection results in diverse weather conditions (i.e., foggy, rainy, snowy) using\nreal-world images from Seeing Through Fog dataset [2]. As shown in Fig. 13,'\ntext from MonoWAD: Weather-Adaptive Diffusion Model for Robust Monocular 3D Object Detection\nauthored by youngmin oh, hyung-il kim, seong tae kim, jung uk kim", 'title': 'monowad: weather-adaptive diffusion model for robust monocular 3d object detection', 'author': ['youngmin oh', 'hyung-il kim', 'seong tae kim', 'jung uk kim'], 'date': 20240723.0, 'source': 'arxiv'}, {'text': "'duce the weather codebook to generate a weather-reference feature that contains\nknowledge about reference weather in a given scene. The reference weather acts\nas a guide, indicating the degree of weather improvement required. Since the'\ntext from MonoWAD: Weather-Adaptive Diffusion Model for Robust Monocular 3D Obj

In [5]:
# 